# Explainability & Business Insights

This notebook documents the business-oriented evaluation stage of the
Campania Financial & ESG Forecasting project.

After selecting Random Forest and generating company-level predictions,
the analysis investigates whether model performance is stable across
heterogeneous company groups and translates the forecasts into
decision-oriented indicators.

The original company-level dataset is proprietary and cannot be redistributed.
Reported values below come from the original analysis, while the executable
examples use small synthetic tables.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Model performance by company size

The original analysis showed that Random Forest error varied more strongly
by firm size than by geography or macro-sector.

Reported EBITDA MSE by company size:

- **Micro:** 0.00076
- **Small:** 0.00219
- **Medium/Large:** 0.00505

The model therefore achieved its lowest error on micro firms and a higher,
although still limited, error on larger firms.

In [ ]:
mse_by_size = pd.DataFrame(
    {
        "company_size": ["micro", "small", "medium/large"],
        "mse": [0.00076, 0.00219, 0.00505],
    }
)

mse_by_size

In [ ]:
ax = mse_by_size.plot(
    x="company_size",
    y="mse",
    kind="bar",
    legend=False,
    figsize=(7, 4),
)

ax.set_title("Reported EBITDA MSE by Company Size")
ax.set_xlabel("")
ax.set_ylabel("Mean Squared Error")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 2. Geography and macro-sector

The original evaluation also grouped squared errors by:

- operational province;
- economic macro-sector.

Unlike company size, the resulting MSE values were broadly comparable
across Campania's five provinces and across the five macro-sectors.

This supports the interpretation that the selected model behaved relatively
consistently across geography and economic activity, while firm size captured
a larger share of performance heterogeneity.

## 3. Employment-growth classification

Employment dynamics were evaluated over the 2021–2024 period using a
three-year Compound Annual Growth Rate (CAGR).

The classification rule was:

- **growing:** CAGR > +10%
- **stable:** CAGR between −10% and +10%
- **declining:** CAGR < −10%

In [ ]:
def employment_cagr(start_employees, end_employees, years=3):
    if (
        pd.isna(start_employees)
        or pd.isna(end_employees)
        or start_employees <= 0
        or years <= 0
    ):
        return np.nan

    return (end_employees / start_employees) ** (1 / years) - 1


def classify_growth(cagr):
    if pd.isna(cagr):
        return np.nan
    if cagr > 0.10:
        return "growing"
    if cagr < -0.10:
        return "declining"
    return "stable"

### Reported distribution by macro-sector

The following table reproduces the company counts reported in the project.

In [ ]:
employment_distribution = pd.DataFrame(
    {
        "Macro-sector": [
            "Trade and Consumer Services",
            "Finance, Real Estate and Collective Services",
            "Industry and Construction",
            "Knowledge Economy and Advanced Services",
            "Networks, Logistics and Infrastructure",
        ],
        "declining": [180, 26, 241, 47, 59],
        "growing": [1003, 133, 1038, 268, 266],
        "stable": [1569, 283, 1831, 295, 421],
    }
)

employment_distribution

The largest absolute numbers of growing firms were observed in
**Industry and Construction** and **Trade and Consumer Services**, which were
also among the most represented macro-sectors in the dataset.

## 4. Economic–Sustainability Positioning Matrix

The final business-oriented visualization combines:

- predicted **EBITDA** as a proxy for future economic performance;
- predicted **ESG score** as a proxy for future sustainability performance.

Median thresholds on both axes divide companies into four interpretable
performance profiles:

- high economic / high sustainability;
- high economic / low sustainability;
- low economic / high sustainability;
- low economic / low sustainability.

This is a decision-support visualization rather than an investment
recommendation.

In [ ]:
rng = np.random.default_rng(42)

positioning_demo = pd.DataFrame(
    {
        "predicted_ebitda": rng.beta(2.0, 4.0, size=250),
        "predicted_esg": rng.beta(2.7, 2.1, size=250),
        "macro_sector": rng.choice(
            [
                "Industry and Construction",
                "Trade and Consumer Services",
                "Networks, Logistics and Infrastructure",
                "Knowledge Economy and Advanced Services",
                "Finance, Real Estate and Collective Services",
            ],
            size=250,
        ),
    }
)

x_threshold = positioning_demo["predicted_ebitda"].median()
y_threshold = positioning_demo["predicted_esg"].median()

positioning_demo["profile"] = np.select(
    [
        (positioning_demo["predicted_ebitda"] >= x_threshold)
        & (positioning_demo["predicted_esg"] >= y_threshold),

        (positioning_demo["predicted_ebitda"] >= x_threshold)
        & (positioning_demo["predicted_esg"] < y_threshold),

        (positioning_demo["predicted_ebitda"] < x_threshold)
        & (positioning_demo["predicted_esg"] >= y_threshold),
    ],
    [
        "high economic / high sustainability",
        "high economic / low sustainability",
        "low economic / high sustainability",
    ],
    default="low economic / low sustainability",
)

positioning_demo.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    positioning_demo["predicted_ebitda"],
    positioning_demo["predicted_esg"],
    alpha=0.6,
)

ax.axvline(x_threshold, linestyle="--")
ax.axhline(y_threshold, linestyle="--")

ax.set_title("Economic–Sustainability Positioning — Synthetic Demo")
ax.set_xlabel("Predicted Economic Performance (EBITDA)")
ax.set_ylabel("Predicted Sustainability Performance (ESG)")
plt.tight_layout()
plt.show()

## 5. Interpretation of the original results

In the original dataset, the high-EBITDA / high-ESG region contained a notable
concentration of firms from **Industry and Construction**.

The analysis also identified firms with strong predicted economic performance
but comparatively weaker ESG outcomes, including cases in
**Trade and Consumer Services**.

These patterns were used to demonstrate how predictive analytics can support
structured comparison across heterogeneous private companies.

## Key Takeaways

The business-analysis layer complements predictive accuracy with structured
interpretation.

It shows that:

- model error differs meaningfully by company size;
- performance is comparatively stable across geography and macro-sector;
- employment dynamics provide an additional forward-looking company profile;
- predicted EBITDA and ESG can be combined into an interpretable
  two-dimensional positioning framework.

## Next Step

The final notebook documents the GenAI-assisted workflow developed to recover
missing operational-province information.